# HW3 - Q3 [35 pts]

## Important Notices

<div class="alert alert-block alert-danger">
    WARNING: <strong>REMOVE</strong> any print statements added to cells with "#export" that are used for debugging purposes before submitting because they will crash the autograder in Gradescope. Any additional cells can be used for testing purposes at the bottom. 
</div>

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> remove any comment that says "#export" because that will crash the autograder in Gradescope. We use this comment to export your code in these cells for grading.
</div>

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> import any additional libraries into this notebook.
</div>

All instructions, code comments, etc. in this notebook **are part of the assignment instructions**. That is, if there are instructions about completing a task in this notebook, that task is not optional.  

<div class="alert alert-block alert-info">
    You <strong>must</strong> implement the following functions in this notebook to receive credit.
</div>

`user()` - 1 point

`trip_statistics()` - 3 points

`busiest_hour()` - 5 points

`most_freq_pickup_locations()` - 5 points

`avg_trip_distance_and_duration()` - 6 points

`most_freq_peak_hour_fares()` - 10 points

Each function will be auto-graded using different sets of parameters or data, to ensure that values are not hard-coded.  You may assume we will only use your code to work with data from the NYC-TLC dataset during auto-grading.

In addition, you will also submit the resulting output csv from most_freq_peak_hour_fares() as q3_output_large.csv.

`q3_output_large.csv` - 5 points

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> remove or modify the following utility functions:
</div>

`load_data()`

`main()`

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> modify the below cell. It contains the function for loading data and all imports, and the function for running your code.
</div>

In [2]:
#export
from pyspark.sql.functions import *
from pyspark.sql import *

Calculation started (calculation_id=8cd0ba11-8097-0a36-14de-7ac388359013) in (session=04d0ba0b-fb29-4475-78f0-855fca1f471d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


In [3]:
#### DO NOT CHANGE ANYTHING IN THIS CELL ####

def load_data(size='small'):
    # Loads the data for this question. Do not change this function.
    # This function should only be called with the parameter 'small' or 'large'
    
    if size != 'small' and size != 'large':
        print("Invalid size parameter provided. Use only 'small' or 'large'.")
        return
    
    input_bucket = "s3://cse6242-hw3-q3"
    
    # Load Trip Data
    trips_path = '/'+size+'/yellow_tripdata*'
    trips = spark.read.csv(input_bucket + trips_path, header=True, inferSchema=True)
    
    # Load Zone Data
    zones_path = '/'+size+'/taxi*'
    zones = spark.read.csv(input_bucket + zones_path, header=True, inferSchema=True)
    
    return trips, zones
    
def main(size, bucket):
    # Runs your functions
    trips, zones = load_data(size=size)
    
    print("User:", user())
    print()
    
    print("Trip Statistics:")
    ts = trip_statistics(trips)
    ts.show()
    print()
    
    print("Busiest Hour:")
    bh = busiest_hour(trips)
    bh.show(24)
    print()
    
    print("Most Frequent Pickup Locations:")
    mfpl = most_freq_pickup_locations(trips)
    mfpl.show()
    print()
    
    print("Average Trip Distance and Duration:")
    atdd = avg_trip_distance_and_duration(trips)
    atdd.show(n=24)
    print()
    
    print("Most Frequent Peak Hour Fares:")
    mfphf = most_freq_peak_hour_fares(trips, zones)
    mfphf.show()
    mfphf.coalesce(1).write.option("header","true").mode("overwrite").csv('{}/output_{}'.format(bucket, size))

Calculation started (calculation_id=8cd0ba11-90aa-5d97-9c47-ceb7749d3648) in (session=04d0ba0b-fb29-4475-78f0-855fca1f471d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


# Implement the below functions for this assignment:
<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> change any function inputs or outputs, and ensure that the dataframes your code returns align with the schema definitions commented in each function. Do <strong>NOT</strong> remove the #export comment from each of the code blocks either. This can prevent your code from being converted to a python file.
</div>

## 3.1 [1 pt] Update the `user()` function
This function should return your GT username, eg: gburdell3

In [4]:
#export
def user():
    return 'nheppermann3'

Calculation started (calculation_id=d6d0ba11-a398-67f6-f9d4-a988715d1954) in (session=04d0ba0b-fb29-4475-78f0-855fca1f471d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


## 3.2 [3 pts] Update the `trip_statistics()` function
This function performs exploratory data analysis on the column trip_distance. Compute basic statistics (count, mean, stdev, min, max) for trip_distance. 

Example output formatting:

```
+-------+------------------+
|summary|     trip_distance|
+-------+------------------+
|  count|           xxxxxxx|
|   mean|           xxxxxxx|
| stddev|           xxxxxxx|
|    min|           xxxxxxx|
|    max|           xxxxxxx|
+-------+------------------+
```
Tip: Is there a PySpark Dataframe function you can use to solve this in a single line?

In [20]:
#export
from pyspark.sql import functions as F
def trip_statistics(trips):
    
    trip_distance_name = 'trip_distance'
    
    #df = trips.agg(
    #    F.count(trip_distance_name).alias('count'),
    #    F.mean(trip_distance_name).alias('mean'),
    #    F.stddev(trip_distance_name).alias('stddev'),
    #    F.min(trip_distance_name).alias('min'),
    #    F.max(trip_distance_name).alias('max'),
    #)
    
    df = trips.select(trip_distance_name).summary("count", "mean", "stddev", "min", "max")
    return df

Calculation started (calculation_id=84d0ba17-83c6-2dc7-03bb-1e97cf6f1f69) in (session=04d0ba0b-fb29-4475-78f0-855fca1f471d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


## 3.3 [5 pts] Update the `busiest_hour()` function

Determine the hour of the day with the highest number of trips. Display the hour (0-23) and the corresponding trip count.

Returns a PySpark DataFrame with a single row showing the hour with the highest trip count and the corresponding number of trips. Use column names: `hour`, `trip_count`.

Example output formatting:

```
+----+----------+
|hour|trip_count|
+----+----------+
|  xx|    xxxxxx|
+----+----------+
```

In [36]:
#export
def busiest_hour(trips):
    
    
    # |-- tpep_pickup_datetime: string (nullable = true)
    tpep_pickup_datetime_name = 'tpep_pickup_datetime'
    hour_name = 'hour'
    trip_count_name = 'trip_count'
    
    df = trips.withColumn(hour_name, F.hour(F.to_timestamp(F.col(tpep_pickup_datetime_name), "yyyy-MM-dd HH:mm:ss")))
    df = df.select(tpep_pickup_datetime_name, hour_name)
    df = df.groupBy(hour_name).agg(F.count(hour_name).alias(trip_count_name))
    df = df.orderBy(F.desc(trip_count_name)).limit(1)
    
    return df

Calculation started (calculation_id=a4d0ba1e-e64e-b829-2bcc-1dc47d10d8d2) in (session=04d0ba0b-fb29-4475-78f0-855fca1f471d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


## 3.4 [5 pts] Update the `most_freq_pickup_locations()` function
Top 10 Most Frequent Pickup Locations

Identify the top 10 pickup locations (by `PULocationID`) with the highest number of trips. Display the location IDs along with their corresponding trip counts.

Return a PySpark DataFrame with the top 10 rows ordered by `trip_count` in descending order. Use column names: `PULocationID`, `trip_count`.

Expected Output:

A table with 10 rows listing the `PULocationID` values and the number of trips observed for each, ordered from most to least frequent.

Example output formatting:
```
+------------+----------+
|PULocationID|trip_count|
+------------+----------+
|         xxx|    xxxxxx|
|         xxx|    xxxxxx|
|         xxx|    xxxxxx|
|         xxx|    xxxxxx|
|         ...|    ......|
+------------+----------+
```

In [38]:
#export
def most_freq_pickup_locations(trips): 
    PULocationID_name = 'PULocationID'
    trip_count_name = 'trip_count'

    df = trips.groupBy(PULocationID_name).agg(F.count(PULocationID_name).alias(trip_count_name))
    df = df.orderBy(F.desc(trip_count_name)).limit(10)
    
    return df

Calculation started (calculation_id=74d0ba20-a970-6c09-3a0e-7dccc6549e28) in (session=04d0ba0b-fb29-4475-78f0-855fca1f471d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


## 3.5 [6 pts] Update the `avg_trip_distance_and_duration()` function
Average Trip Distance and Duration by Hour

Calculate the average trip distance and average trip duration by pickup hour (0-23), using `tpep_pickup_datetime` for the hour. Display the hour along with the corresponding averages.

Compute trip duration in minutes (difference between drop-off time and pickup time).

Exclude rows where `tpep_pickup_datetime` or `tpep_dropoff_datetime` is null and where `trip_distance` <= 0. No additional outlier filtering or rounding is required.

Note: You can use `unix_timestamp` to help with calculating the duration.

Expected Output:

A table with 24 rows (hours 0-23) showing each hour ordered ascending along with the average trip distance and average trip duration for that hour. Use column names: `hour`, `avg_trip_distance`, `avg_trip_duration`.

Example output formatting:
```
+----+------------------+------------------+
|hour| avg_trip_distance| avg_trip_duration|
+----+------------------+------------------+
|   0|           xxxxxxx|           xxxxxxx|
|   1|           xxxxxxx|           xxxxxxx|
|   2|           xxxxxxx|           xxxxxxx|
|   3|           xxxxxxx|           xxxxxxx|
| ...|               ...|               ...|
|  23|           xxxxxxx|           xxxxxxx|
+----+------------------+------------------+
```

In [51]:
#export
def avg_trip_distance_and_duration(trips):
    
    # |-- tpep_pickup_datetime: string (nullable = true)
    tpep_pickup_datetime_name = 'tpep_pickup_datetime'
    tpep_dropoff_datetime_name = 'tpep_dropoff_datetime'
    trip_distance_name = 'trip_distance'
    
    # Exclude rows where tpep_pickup_datetime or tpep_dropoff_datetime is null and where trip_distance <= 0.
    filtered_df = trips.dropna(subset=[tpep_pickup_datetime_name, tpep_dropoff_datetime_name])
    df = filtered_df.filter(filtered_df[trip_distance_name] > 0.0)
    
    #df = df.limit(10) #to remove
    
    # convert times to timestamps
    tpep_pickup_name = 'tpep_pickup'
    tpep_dropoff_name = 'tpep_dropoff'
    tformat_str = 'yyyy-MM-dd HH:mm:ss'
    
    df = df.withColumn(tpep_pickup_name, F.to_timestamp(F.col(tpep_pickup_datetime_name), "yyyy-MM-dd HH:mm:ss"))
    df = df.withColumn(tpep_dropoff_name, F.to_timestamp(F.col(tpep_dropoff_datetime_name), "yyyy-MM-dd HH:mm:ss"))
    
    # get duration in mins
    secs_per_min = 60
    duration_mins_name = 'duration_mins'
    df = df.withColumn(
        duration_mins_name, 
        (F.col(tpep_dropoff_name).cast('long') - F.col(tpep_pickup_name).cast('long')) / secs_per_min
    )
    
    hour_name = 'hour'
    df = df.withColumn(hour_name, F.hour(F.col(tpep_pickup_name)))

    avg_trip_distance_name = 'avg_trip_distance'
    avg_trip_duration_name = 'avg_trip_duration'

    df = df.groupBy(hour_name).agg(
        F.avg(trip_distance_name).alias(avg_trip_distance_name),
        F.avg(duration_mins_name).alias(avg_trip_duration_name)
    )
    
    df = df.select(
        hour_name, 
        avg_trip_distance_name,
        avg_trip_duration_name
    ).orderBy(F.asc(hour_name))
    
    
    #df = df.select(
    #    hour_name, 
    #    tpep_pickup_datetime_name, tpep_pickup_name,
    #    tpep_dropoff_datetime_name, tpep_dropoff_name,
    #    trip_distance_name
    #)
    #trip_count_name = 'trip_count'

    #df = trips.withColumn(hour_name, F.hour(F.to_timestamp(F.col(tpep_pickup_datetime_name), "yyyy-MM-dd HH:mm:ss")))
    #df = df.select(tpep_pickup_datetime_name, hour_name)
    #df = df.groupBy(hour_name).agg(F.count(hour_name).alias(trip_count_name))
    #df = df.orderBy(F.desc(trip_count_name)).limit(1)
    
    
    return df

Calculation started (calculation_id=c8d0ba2d-3a66-523b-2f81-1fb7ace26959) in (session=04d0ba0b-fb29-4475-78f0-855fca1f471d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


## 3.6 [10 pts] Update the `most_freq_peak_hour_fares()` function
Top 10 Most Frequent Routes During Peak Hours

Identify the top 10 most frequent routes (combinations of `PULocationID` and `DOLocationID`) during peak hours and return route-level statistics. For this question:

- Peak-hour windows are 7:00 - 8:59 (morning) and 16:00 - 18:59 (evening). Consider trips whose pickup hour falls within these windows.
- Exclude rows where `PULocationID` or `DOLocationID` is null.
- Exclude routes where `PULocationID` equals `DOLocationID` (a valid route must have different pickup and drop-off locations).

For each route, compute:

- `trip_count`: number of trips for that route during peak hours, and
- `avg_total_fare`: the average of the trip total_amount for that route, rounded to two decimal places.

Join the route results with the provided zones dataset to include human-readable zone names for pickup and dropoff. Return a PySpark DataFrame with the following columns (in this order):

`PULocationID`, `PUZone`, `DOLocationID`, `DOZone`, `trip_count`, `avg_total_fare`.

Expected Output:

A table with 10 rows showing the top 10 routes during peak hours ordered by `trip_count` descending. If multiple routes share the same `trip_count`, any stable ordering among them is acceptable as long as results are ordered by `trip_count` descending.

Example output formatting:
```
+------------+------+------------+------+----------+--------------+
|PULocationID|PUZone|DOLocationID|DOZone|trip_count|avg_total_fare|
+------------+------+------------+------+----------+--------------+
|xxx         |xxx   |xxx         |xxx   |xxx       |xx.xx         |
|xxx         |xxx   |xxx         |xxx   |xxx       |xx.xx         |
|xxx         |xxx   |xxx         |xxx   |xxx       |xx.xx         |
|...         |...   |...         |...   |...       |...           |
+------------+------+------------+------+----------+--------------|

```

In [9]:
#export
def most_freq_peak_hour_fares(trips, zones):
    return

Calculation started (calculation_id=bed0ba11-c992-4ad4-28d7-c7d0c396939b) in (session=04d0ba0b-fb29-4475-78f0-855fca1f471d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


## 3.7 [5 pts] q3_output_large.csv
The CSV output from running `most_freq_peak_hour_fares(trips, zones)` on the large dataset.

- Run the `main` function with `size='large'` to generate the file in the S3 bucket you created in the AWS setup.
- Download the generated file from S3 and rename it to `q3_output_large.csv`.

<div class="alert alert-block alert-info">
<h2>Submission</h2>

<p>Once you have finished coding, you can export the notebook from <code>Notebook Explorer</code> by selecting your notebook and clicking <code>Export File</code> from the Actions dropdown.</p>

<p>Submit this notebook (q3.ipynb) along with your CSV file (q3_output_large.csv) to Gradescope.</p>
</div>

<div class="alert alert-block alert-info">
  <h2>Testing</h2>

  <p>You may use the cell below for any additional testing; however, any code written here will not be run or used during grading.</p>

  <ul>
    <li>You can run the <code>main</code> function on different dataset sizes to test your work, or run the functions individually as shown in the examples.</li>
  </ul>
</div>

In [10]:
trips, zones = load_data('small')

Calculation started (calculation_id=2ed0ba11-cf1d-2897-33a5-1fb39864dea4) in (session=04d0ba0b-fb29-4475-78f0-855fca1f471d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


In [13]:
# my debug
trips.printSchema()
trips.show()

Calculation started (calculation_id=26d0ba13-1631-693f-5c07-47627efdbfa1) in (session=04d0ba0b-fb29-4475-78f0-855fca1f471d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------

In [14]:
# my debug
zones.printSchema()
zones.show()

Calculation started (calculation_id=a0d0ba13-2373-16e6-4ac1-35b3cd234461) in (session=04d0ba0b-fb29-4475-78f0-855fca1f471d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|    

In [21]:
ts = trip_statistics(trips)
ts.show()

Calculation started (calculation_id=c0d0ba17-9053-68d7-4271-138c47f2ca57) in (session=04d0ba0b-fb29-4475-78f0-855fca1f471d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
+-------+------------------+
|summary|     trip_distance|
+-------+------------------+
|  count|           7667792|
|   mean|2.8010838491705305|
| stddev| 3.737529402987038|
|    min|               0.0|
|    max|             831.8|
+-------+------------------+



In [37]:
bh = busiest_hour(trips)
bh.show()

Calculation started (calculation_id=48d0ba1e-eff4-953e-cfa1-401c59dde995) in (session=04d0ba0b-fb29-4475-78f0-855fca1f471d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
+----+----------+
|hour|trip_count|
+----+----------+
|  18|    514036|
+----+----------+



In [39]:
mfpl = most_freq_pickup_locations(trips)
mfpl.show()

Calculation started (calculation_id=74d0ba20-e037-08dc-778d-98028859c597) in (session=04d0ba0b-fb29-4475-78f0-855fca1f471d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
+------------+----------+
|PULocationID|trip_count|
+------------+----------+
|         237|    332473|
|         236|    323008|
|         161|    312392|
|         162|    277166|
|         230|    263646|
|         186|    260712|
|          48|    240903|
|         170|    238978|
|         234|    237648|
|         142|    235144|
+------------+----------+



In [52]:
atdad = avg_trip_distance_and_duration(trips)
atdad.show()

Calculation started (calculation_id=fcd0ba2d-43d0-86f7-750d-0bacabaeddd4) in (session=04d0ba0b-fb29-4475-78f0-855fca1f471d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
+----+------------------+------------------+
|hour| avg_trip_distance| avg_trip_duration|
+----+------------------+------------------+
|   0|3.3860105571163275|16.066586872624416|
|   1| 3.169274824663553|15.838700840351354|
|   2| 3.051906061250731| 15.66075140516562|
|   3|  3.29463880946192|16.983775452733692|
|   4| 4.100486134257711| 18.15875191653886|
|   5| 4.325835327712818|16.922907261354386|
|   6| 3.244580313544074|17.653774046469962|
|   7|2.7927728882312812|19.078441754916735|
|   8|2.5191803806457638|17.628541940021922|
|   9|2.4761999861907196|16.901794655803375|
|  10|2.5444368670403246|16.403171316182906|
|  11| 2.453481901274788|16.630468273946494|
|  12|2.5596964691879713| 16.32168134266004|
|  13|2.7299724685453013| 16.59014836951219|
|  14|2.8004283000039667| 17.19750086273839|
|  15|2.7461276683325773|17.548222938599793|
|  16|2.8595799849684864|17.849531710701257|
|  17|2.6456560803738056|17.295558073532057|
|  18| 2.535206925898749|  16.36

In [ ]:
main('large', 's3://cse6242-nheppermann3')